# Sigiriya SinLlama legacy RAG comparison v1
Select a T4 GPU runtime, then run all cells. Upload the ZIP built with
`python -B -m scripts.sinllama_legacy_rag_comparison_v1 bundle`
from the project directory. All nine runs use the bundled Neo4j evidence and one Transformers 4-bit SinLlama model.

A uses the v4 Sinhala system prompt and decoding. B uses the original RAG prompt with v4 decoding. C uses the original prompt, decoding, and answer-boundary rules.

In [3]:
%pip install -q transformers==4.57.1 accelerate==1.10.1 bitsandbytes==0.48.2

In [5]:
from io import BytesIO
from pathlib import Path
from tempfile import mkdtemp
from zipfile import ZipFile

import torch
from google.colab import files

assert torch.cuda.is_available(), "Select a GPU runtime before running the experiment."
print(torch.cuda.get_device_name())
bundle_name = "sigiriya_sinllama_legacy_rag_runtime_v1.zip"
uploaded = files.upload()

# Find the uploaded key, handling potential duplicate renaming by Colab (e.g., 'sigiriya_sinllama_legacy_rag_runtime_v1 (2).zip')
uploaded_key = next((k for k in uploaded if k.startswith("sigiriya_sinllama_legacy_rag_runtime_v1")), None)
assert uploaded_key is not None, f"Upload {bundle_name}"

workspace = Path(mkdtemp(prefix="sigiriya_legacy_rag_v1_", dir="/content"))
with ZipFile(BytesIO(uploaded[uploaded_key])) as archive:
    archive.extractall(workspace)

Tesla T4


Saving sigiriya_sinllama_legacy_rag_runtime_v1.zip to sigiriya_sinllama_legacy_rag_runtime_v1 (3).zip


In [6]:
import os
import subprocess
import sys

name = "sinllama_legacy_rag_comparison_v1"
output = workspace / "results" / name
subprocess.run(
    [sys.executable, "-B", "-m", f"scripts.{name}", "gpu", "--output", str(output)],
    cwd=workspace, env={**os.environ, "PYTHONIOENCODING": "utf-8"}, check=True,
)

CompletedProcess(args=['/usr/bin/python3', '-B', '-m', 'scripts.sinllama_legacy_rag_comparison_v1', 'gpu', '--output', '/content/sigiriya_legacy_rag_v1_k46b0nde/results/sinllama_legacy_rag_comparison_v1'], returncode=0)

Review the cleaned model answer for relevance, factual correctness, and completeness. Check that Q01 preserves the source's uncertainty, Q03 covers the frescoes and Mirror Wall, and Q05 explains underground channels, gravity, and water pressure while preserving uncertainty. The CSV leaves human assessments blank. Automated verifier decisions and fallback flags appear in separate columns.

In [7]:
import json
import pandas as pd

report = json.loads((output / f"{name}.json").read_text(encoding="utf-8"))
assert {(row["question_id"], row["experiment"]) for row in report["results"]} == {
    (question, arm) for question in ("Q01", "Q03", "Q05") for arm in ("A", "B", "C")
}
review = pd.read_csv(output / f"{name}_human_review.csv", keep_default_na=False)
display(review.drop(columns=["sinhala_question", "cleaned_model_answer"]))

,question,experiment,relevant_human,factually_correct_human,complete_human,repetition_detected,strict_accepted,strict_fallback,experimental_accepted,experimental_fallback
0,Q01,A,,,,False,True,False,True,False
1,Q01,B,,,,False,False,True,False,True
2,Q01,C,,,,False,False,True,False,True
3,Q03,A,,,,False,False,True,False,True
4,Q03,B,,,,False,False,True,False,True
5,Q03,C,,,,False,False,True,False,True
6,Q05,A,,,,False,False,True,False,True
7,Q05,B,,,,False,True,False,True,False
8,Q05,C,,,,False,False,True,False,True


In [8]:
from shutil import make_archive

archive = make_archive(str(workspace / f"{name}_outputs"), "zip", output)
files.download(archive)

<IPython.core.display.Javascript object>

<IPython.core.display.Javascript object>